# Backfill sources (manual, capped, resumable)
Run from the `backfill` job on days **without** the full daily run (Open-Meteo free tier: <10k calls/day).

| mode | What | Lands in | Tags (`data_source` / `request_mode`) | Cost |
| --- | --- | --- | --- | --- |
| `recent` | Forecast API past 11 days + today, all grid cells | `open_meteo_forecast_backfill/run_date=` | `open_meteo_forecast` / `backfill_recent` | 8,060 calls, one run |
| `history` | ERA5 reanalysis for a sample of cells (GDACS-event cells + random per country) | `open_meteo_era5/year=`, sample in `reference/history_cells/` | `open_meteo_era5` / `backfill_history` | ~52 calls per cell for 2 years; several runs |

Each run stops cleanly at `max_calls`; re-run with the **same** `run_date` (recent) or the same
`history_start` / `history_end` (history) to resume. Code: `history_lib.py`, `ingest_lib.py`.

In [0]:
import datetime as dt, json
import history_lib as hist  # same folder
import ingest_lib as lib

dbutils.widgets.text("catalog", "weather_forecast_warnings_comms")
dbutils.widgets.text("schema", "bronze")
dbutils.widgets.text("volume", "landing")
dbutils.widgets.dropdown("mode", "recent", ["recent", "history"])
dbutils.widgets.text("countries", "PER,ECU,COL,PAN,CRI")
dbutils.widgets.text("max_calls", "9000")            # weighted Open-Meteo calls for this run
dbutils.widgets.text("run_date", "")                 # recent: blank = today; reuse to resume
dbutils.widgets.text("history_start", "2024-11-01")  # history: fixed range so resumes match
dbutils.widgets.text("history_end", "2026-10-31")    # ERA5 lags ~7 days
dbutils.widgets.text("per_country", "60")            # random cells per country (plus GDACS cells)

w = dbutils.widgets.get
ROOT = f"/Volumes/{w('catalog')}/{w('schema')}/{w('volume')}"
MODE, MAX_CALLS = w("mode"), float(w("max_calls"))
COUNTRIES = [c.strip() for c in w("countries").split(",") if c.strip()]
RUN_DATE = w("run_date") or dt.datetime.now(dt.timezone.utc).date().isoformat()
START, END, PER_COUNTRY = w("history_start"), w("history_end"), int(w("per_country"))
INGESTED_AT = dt.datetime.now(dt.timezone.utc).isoformat()
summary = {"mode": MODE, "max_calls": MAX_CALLS}
print(ROOT, summary, COUNTRIES)

In [0]:
grid = spark.read.json(f"{ROOT}/reference/grid_cells").where(f"iso3 IN ({','.join(repr(c) for c in COUNTRIES)})")

if MODE == "recent":
    pts = grid.select("cell_id", "lat", "lon").orderBy("cell_id").collect()
    summary.update(run_date=RUN_DATE, **lib.land_forecast(ROOT, RUN_DATE, pts, 11, 1, INGESTED_AT, MAX_CALLS,
                                                          request_mode="backfill_recent", on_budget="stop"))
else:
    cells = hist.select_history_cells(spark, ROOT, COUNTRIES, START, END, INGESTED_AT, PER_COUNTRY)
    display(cells.selectExpr("iso3", "explode(sample_reasons) AS reason").groupBy("iso3", "reason").count()
            .orderBy("iso3", "reason"))
    days = (dt.date.fromisoformat(END) - dt.date.fromisoformat(START)).days + 1
    rows = cells.select("cell_id", "lat", "lon").orderBy("cell_id").collect()
    summary.update(history_start=START, history_end=END, history_cells=len(rows),
                   planned_calls_total=round(len(rows) * days / 14))
    summary.update(hist.land_era5(ROOT, rows, START, END, INGESTED_AT, MAX_CALLS))

print(json.dumps(summary))
dbutils.notebook.exit(json.dumps(summary))